# TN2211 Session 10: Noise

## Instruments


In [ ]:
import sys
sys.path.append("../drivers/")
from tn2211_drivers import *
import glob
import matplotlib.pyplot as plt
import math
import numpy as np
import time
from scipy.optimize import curve_fit
from scipy.signal import hilbert

In [ ]:
import pyvisa
rm = pyvisa.ResourceManager()
rm.list_resources()

In [ ]:
scope = Scope("SDS")
gen = Generator("SDG")

## Step 1: Building the optical transceiver and finding its limits


In [ ]:
# Activate the channels we will use
scope.write("CHAN1:SWIT ON")
scope.write("CHAN2:SWIT OFF")
scope.write("CHAN3:SWIT OFF")
for i in 1,2,3,4:
    scope.write("FUNC%d OFF" % i)

# We will use Ch4 of the scope connected to the sync out of the 
# generator for triggering but make it not visible
scope.write("CHAN4:SWIT ON")
scope.write("CHAN4:SCAL 2")
scope.write("TRIG:EDGE:SOUR C4")
scope.write("TRIG:EDGE:LEV 1")
scope.write("CHAN4:VIS OFF")

# AC coupling on CH1, since we do not want to see the large offset, 
# only the small signal
scope.write("CHAN1:COUP AC")
scope.write("CHAN1:BWL FULL")
scope.write("CHAN1:OFFS 0")
scope.write("CHAN1:SCALE 2e-3")

scope.write("TIM:SCAL .05e-3")
scope.write("ACQ:MDEP 10k")

# Set up the generators
gen.write("C1:BSWV WVTP,SQUARE,FRQ,10e3,AMP,0.1,OFST,2")
gen.write("C2:BSWV WVTP,DC,OFST,5")
gen.write("C1:OUTP ON")
gen.write("C2:OUTP ON")
gen.write("C1:SYNC ON,TYPE,CH1")

If your circuit is built correctly, you should already see a signal on the screen!

You can confirm that this is a signal from the light being transmitted between the two diodes by blocking the light: try putting a piece of paper or your student card between the diodes: you should see that the signal disappears!

In [ ]:
scope.get_screenshot()

Grab a trace for analysis you can do in Level 2, make a note of the file name in your logbook and write in your logbook a description of what that file measured. Doing so, you will later be able to know which file is which.

In [ ]:
t,v = scope.get_trace(1)
plt.plot(t,v)
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.show()

Lower the PTP amplitude until you estimate that the SNR = 1:

In [ ]:
v_ptp =  # fill in a ptp amplitude to try
gen.set_amplitude(1,v_ptp)

In [ ]:
t,v = scope.get_trace(1)
plt.plot(t,v)
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("v_ptp = %.1f mV" % (v_ptp*1e3))
plt.show()

Pinching the wire:

* first with the timebase setting above
* then try zooming in and zooming out to understand better what you see

In [ ]:
scope.get_screenshot()

In [ ]:
scope.get_screenshot()

In [ ]:
scope.get_screenshot()

## Step 2: Removing aliasing noise by limiting bandwidth

For this step, we will use a 10 mV signal on the transmitter diode and set the scope to a 2 mV scale and a 50 us/div timebase:

In [ ]:
gen.set_amplitude(1, 10e-3)
scope.write("CHAN1:SCALE 2e-3")
scope.write("TIM:SCAL .05e-3")

First start with full bandwidth:

In [ ]:
scope.write("CHAN1:BWL FULL")
t,v = scope.get_trace(1)
plt.plot(t,v)
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("Full bandwidth")
plt.show()

In [ ]:
scope.write("CHAN1:BWL 20M")
t,v = scope.get_trace(1)
plt.plot(t,v)
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("Full bandwidth")
plt.show()

Grab a screenshot so that we can document what the sampling rate was of the scope:

In [ ]:
scope.get_screenshot()

## Step 3: Power spectral density of your noise


Configure the instrument settings and define some helper functions:

In [ ]:
gen.set_amplitude(1, 10e-3)
gen.write("C1:BSWV WVTP,SINE")

scope.write("CHAN1:SCALE 4e-3")
scope.write("CHAN1:BWL 20M")
scope.write("FUNC1 ON")
scope.write("FUNC1:OPER FFT")
scope.write("TIM:SCAL 100e-3")
scope.write("ACQ:MDEP 1M")

def set_freq_range(f1, f2):
    center = (f1+f2)/2
    span = f2-f1
    scope.write("FUNC1:FFT:SPAN %e" % span)
    scope.write("FUNC1:FFT:HCEN %e" % center)

def set_fft_averages(n):
    if n == 1:
        scope.write("FUNC1:FFT:MODE NORM")
        return
    if n<4: 
        n = 4 # min allowed number
    if n>1024:
        n = 1024 # max allowed number
    else:
        scope.write("FUNC1:FFT:MODE AVER,%d" % n)

def reset_fft_average():
    scope.write("FUNC1:FFT:RESET")

set_fft_averages(1)
set_freq_range(0,500e3)

The timebase, combined with the memory depth, will set two things:

* The frequency resolution (given by total trace time)
* The Nyquist frequency = sampling_frequency / 2 = total_trace_time / memory_depth / 2

In practice, to calcualte fast FFTs, the scope will truncate the time trace to an power of 2: with a memory depth of 1M on the time trace, it will only use 2^19 points = 524288. So the nyquist frequency will be a bit lower than you would expect purely from the memory dpeth. 

If you want shaper resolution in your PSD at low frequencies, you should increase the time base to a larger number of seconds per division (slower traces), but for a given memory depth, that will also reduce your nyquist frequency, and therefore the maximum frequency you can observe in your spectrum.

If you want to see higher frequencies in your spectrum, you should decrease the time scale. This will increase your maximum frequency but reduce your frequency resolution. 

A time base of 100 ms per division will give you a 1.9 Hz frequency spacing (due to the way the scope does the FFT) and a maximum frequency of 500 kHz. Note that although your frequency point spacing will be 1.9 Hz, due to the digital filtering applied to reduce spectral leakage, the FFT spectral resolution is smeared out a bit to 7.11 Hz, a number called the "resolution bandwidth". 

In [ ]:
scope.write("TIM:SCAL 100e-3")

First, zoom in at low frequencies where our signal is:

In [ ]:
set_freq_range(0,15e3)

To get better resolution of low frequencies, we need to use a slower time base. Let's go there and take some averages to get something close to the "theoretical smooth power spectral density":

In [ ]:
set_fft_averages(10)
reset_fft_average()

You can configure more averages if you want, and you will get a smoother curve, but you then have to wait longer.

Wait until the average reach 10 (you can see it on the screen) and then grab the time trace and the spectrum:

In [ ]:
scope.write("TRIG:STOP")
freq2, spectrum_dBVrms2, unit = scope.get_fft()
t2,v2 = scope.get_trace(1)
scope.write("TRIG:RUN")

Make a note in your logbook of these filenames and what measurement they corresponeded to so that later you can analyze them in more depth in Level 3 if you want.

In [ ]:
# This number is displayed on the screen, but not available via programming :(
# It will change if you change the time base
# This is the value if you have a timebase of 100 ms/div
RBW = 7.11

# We have dB(Vrms) = 10*log10(V^2) which is the power spectrum.
# We want the power spectral density, which people (confusingly) call dBVrms / Hz 
# dBVrms / Hz is mathematically defined as 10*log10(V^2 / RBW) = dBVrms - 10*log10(RBW)

PSD_dBVrms = spectrum_dBVrms - 10*np.log10(RBW)

Take a look at different frequency ranges:

In [ ]:
plt.figure(figsize=(16,4))
plt.plot(freq, PSD_dBVrms)
plt.xlabel("Freq (Hz)")
plt.ylabel("Power spectral density (dBVrms / Hz)")
plt.xlim(0,15e3)

Put the above plot into your R&A report and discuss what features you see. 

At low frequencies we see 50 Hz from the power lines (always!):

In [ ]:
plt.figure(figsize=(16,4))
plt.plot(freq,  PSD_dBVrms)
plt.xlabel("Freq (Hz)")
plt.ylabel("Power spectral density (dBVrms / Hz)")
plt.xlim(0,1e3)
for f in 50,100,150,200,250:
    plt.axvline(f, ls=":", c='gray')

Here, you can see the effect of our limited resolution bandwidth: these peaks are probably really delta functions. You can test this by repeating the measurement with a different time base. 

In [ ]:
plt.figure(figsize=(16,4))
plt.plot(freq, PSD_dBVrms)
plt.xlabel("Freq (Hz)")
plt.ylabel("Power spectral density (dBVrms / Hz)")
plt.xlim(0,500e3)

THis one is a bit slow to display, but is interactive so you can zoom in and out on different parts of the spectrum interactively:

In [ ]:
bokeh_plot(freq, PSD_dBVrms)

Now pinching the wire:

In [ ]:
set_fft_averages(10)
reset_fft_average()
time.sleep(15) # 10 averages at 100 ms time base is about 15 seconds
scope.write("TRIG:STOP")
freq2, spectrum_dBVrms2, unit = scope.get_fft()
t2,v2 = scope.get_trace(1)
scope.write("TRIG:RUN")

RBW = 7.11
PSD_dBVrms2 = spectrum_dBVrms2 - 10*np.log10(RBW)

In [ ]:
plt.figure(figsize=(16,4))
plt.plot(freq, PSD_dBVrms)
plt.plot(freq2, PSD_dBVrms2)
plt.xlabel("Freq (Hz)")
plt.ylabel("Power spectral density (dBVrms / Hz)")
plt.xlim(0,500e3)

## Step 4: Coherent averaging

In [ ]:
# Activate the channels we will use
scope.write("CHAN1:SWIT ON")
scope.write("CHAN2:SWIT OFF")
scope.write("CHAN3:SWIT OFF")
for i in 1,2,3,4:
    scope.write("FUNC%d OFF" % i)

# We will use Ch4 of the scope connected to the sync out of the 
# generator for triggering but make it not visible
scope.write("CHAN4:SWIT ON")
scope.write("CHAN4:SCAL 2")
scope.write("TRIG:EDGE:SOUR C4")
scope.write("TRIG:EDGE:LEV 1")
scope.write("CHAN4:VIS OFF")

# AC coupling on CH1, since we do not want to see the large offset, 
# only the small signal
scope.write("CHAN1:COUP AC")
scope.write("CHAN1:BWL 20M")
scope.write("CHAN1:OFFS 0")
scope.write("CHAN1:SCALE 0.5e-3")

scope.write("TIM:SCAL .05e-3")
scope.write("ACQ:MDEP 10k")

# Set up the generators
gen.write("C1:BSWV WVTP,SQUARE,FRQ,10e3,AMP,0.01,OFST,2")
gen.write("C2:BSWV WVTP,DC,OFST,5")
gen.write("C1:OUTP ON")
gen.write("C2:OUTP ON")
gen.write("C1:SYNC ON,TYPE,CH1")
time.sleep(2) # let outputs settle after setting generators

# Set up the coherent averaging (trace averaging)
scope.write("FUNC1 OFF")
scope.write("FUNC1 ON")
scope.write("FUNC1:OPER AVER")
scope.write("FUNC1:AVER:NUM 128") ## Must be a power of 2. 4 is smallest number allowed, 1024 the largest.
scope.write("FUNC1:SCAL 0.5e-3")

def reset_average():
    # Pretty stupid, there is no reset function via SCPI....
    scope.write("FUNC1 OFF")
    scope.write("FUNC1 ON")
    

In [ ]:
# Must be a power of 2. 4 is smallest number allowed, 1024 the largest.
# 4, 8, 16, 32, 64, 128, 256, 512, 1024 are allowed.
N_averages = 128
scope.write("FUNC1:AVER:NUM %d" % N_averages) 

In [ ]:
scope.write("TRIG:STOP")
t,v = scope.get_trace(1)
t2,v2 = scope.get_trace("F1")
scope.write("TRIG:RUN")

Make notes of the file names and the settings you used in your logbook so you can do further analysis in Level 2. 

In [ ]:
plt.figure(figsize=(16,4))
plt.plot(t,v, label="Single trace")
plt.plot(t2,v2, label="%d averages" % N_averages)
plt.ylabel("Voltage (V)")
plt.xlabel("Time (t)")
plt.legend()

You can use the code above to try more averages

# Level 2

In [ ]:
# Your code here

# Level 3

In [ ]:
# Your code here